# Temporal Multiplex Directed Networks for the Semiconductor Industry

A Temporal Multiplex Directed Network $\mathcal{M}$ is defined as a sequence of layers $L = \{L_1, L_2, \dots, L_M\}$, where each layer represents a different type of interaction (Financial, Supply Chain, etc.) over time steps $t \in \{1, \dots, T\}$.

The state of the network at any time $t$ is represented by a Supra-Adjacency Tensor $\mathcal{A}$: $$\mathcal{A}_{i,j, \alpha}(t)$$
Where: 
$i, j \in \{1, \dots, N\}$ are the semiconductor companies (nodes). 
$\alpha \in \{1, \dots, M\}$ is the specific layer (e.g., $\alpha=1$ for the Financial Layer, $\alpha=2$ for the Supply Chain Layer, $\alpha=3$ for the Ownership Layer). $t$ is the temporal window (e.g., the specific week).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import matplotlib.pyplot as plt
import networkx as nx
from py_scripts.project2 import financial_layer as fl

## Financial Layer

TODO WRITE A SUMMARY OF THE STEPS DONE AND WHAT WAS ACHIEVED


### DATA ACQUISITION

In [3]:
# Sampling: hourly lead-lag failed the time-shuffle placebo test (no detectable 1h
# cross-predictability among liquid semis), so we work at DAILY frequency with a
# longer history — where supplier->customer lead-lag is actually documented.
INTERVAL = "1d"
START = "2021-11-01"  # constrained by GFS (IPO 2021-10); gives ~1,150 common daily bars

foundries = {
    "TSM": "Taiwan Semiconductor Manufacturing Company Limited",
    # "SSNLF" (Samsung) dropped: OTC ADR, 98% stale daily closes — it barely trades
    "INTC": "Intel Corporation",
    "UMC": "United Microelectronics Corporation",
    "GFS": "GlobalFoundries Inc.",
}

fabless_designers = {
    "NVDA": "NVIDIA Corporation",
    "AMD": "Advanced Micro Devices, Inc.",
    "AVGO": "Broadcom Inc.",
    # "ARM" (Arm Holdings) dropped: IPO 2023-09 would cap the common sample at ~580 bars — re-add if trading recency over history
    "QCOM": "QUALCOMM Incorporated",
    "MRVL": "Marvell Technology, Inc.",
    # "ALAB" (Astera Labs) dropped: IPO 2024-03, same short-history problem
}

memory = {
    "MU": "Micron Technology, Inc.",
}

wfe = {
    "ASML": "ASML Holding N.V.",
    "AMAT": "Applied Materials, Inc.",
    "LRCX": "Lam Research Corporation",
    "KLAC": "KLA Corporation",
    "TOELY": "Tokyo Electron Limited",  # OTC ADR but clean at daily frequency (<0.5% stale closes)
    # "ADVNF" (Advantest) dropped: OTC ADR, data only since 2024-03 and 27% stale
    "TER": "Teradyne, Inc.",
    "SNPS": "Synopsys, Inc.",
    "CDNS": "Cadence Design Systems, Inc.",
}

osat_packaging = {
    "ASX": "ASE Technology Holding Co., Ltd.",
    "AMKR": "Amkor Technology, Inc.",
}

analog_auto_power = {
    "TXN": "Texas Instruments Incorporated",
    "ADI": "Analog Devices, Inc.",
    "NXPI": "NXP Semiconductors N.V.",
    "STM": "STMicroelectronics N.V.",  # was "STNE", which is StoneCo (Brazilian fintech), not STMicro
    "ON": "ON Semiconductor Corporation",
    "IFNNY": "Infineon Technologies AG",  # OTC ADR but clean at daily frequency (<0.5% stale closes)
    "MCHP": "Microchip Technology Incorporated"
}

# Organize spheres
spheres = {
    "Foundries": foundries,
    "Fabless Designers": fabless_designers,
    "Memory": memory,
    "WFE (Equipment)": wfe,
    "OSAT & Packaging": osat_packaging,
    "Analog/Auto/Power": analog_auto_power,
}

In [4]:
# Download and visualize each sphere
for sphere_name, tickers_dict in spheres.items():
    data = yf.download(list(tickers_dict.keys()), start=START, interval=INTERVAL, prepost=False, progress=False)['Close']

    df = data.reset_index()
    time_col = df.columns[0]  # 'Date' for daily bars, 'Datetime' for intraday
    fig = px.line(df, x=time_col, y=data.columns,
                  title=f'{sphere_name} - Close Price (since {START}, {INTERVAL} bars)',
                  labels={'value': 'Close Price (USD)', 'variable': 'Ticker'})
    fig.update_layout(xaxis_title=time_col, yaxis_title='Price (USD)')
    fig.show()

In [5]:
# Compute and visualize returns for each sphere
total_returns = pd.DataFrame()
for sphere_name, tickers_dict in spheres.items():
    data = yf.download(list(tickers_dict.keys()), start=START, interval=INTERVAL, prepost=False, progress=False)['Close']
    data = data.replace(0, np.nan).ffill()
    returns = np.log(data / data.shift(1))

    if INTERVAL.endswith('h') or INTERVAL.endswith('m'):
        # Intraday only: the first bar of each trading day spans the overnight/weekend
        # gap, not one bar-length — drop it so only true intra-session returns remain.
        is_session_start = returns.index.to_series().dt.date != returns.index.to_series().shift(1).dt.date
        returns = returns[~is_session_start]
    returns = returns.dropna()

    total_returns = pd.concat([total_returns, returns], axis=1, sort=True)

    df = returns.reset_index()
    time_col = df.columns[0]
    fig = px.line(df, x=time_col, y=returns.columns,
                  title=f'{sphere_name} - Returns (since {START}, {INTERVAL} bars)',
                  labels={'value': 'Returns', 'variable': 'Ticker'})
    fig.update_layout(xaxis_title=time_col, yaxis_title='Returns')
    fig.show()

# Keep only bars where every asset traded (cross-sphere concat can leave NaN rows
# when listings have slightly different calendars).
total_returns = total_returns.dropna()
print("total_returns:", total_returns.shape)

total_returns: (1179, 27)


In [6]:
# Data-quality diagnostic for the full universe at the working interval:
# catches empty tickers (all-NaN columns kill entire spheres via dropna) and
# stale OTC listings (repeated closes) before they poison the pipeline.
all_tickers = [t for d in spheres.values() for t in d]
prices = yf.download(all_tickers, start=START, interval=INTERVAL, prepost=False, progress=False)['Close']

all_nan = prices.columns[prices.isna().all()].tolist()
coverage = prices.notna().mean()
stale = (prices.diff() == 0).mean()

report = pd.DataFrame({'coverage': coverage, 'stale_frac': stale,
                       'first_valid': prices.apply(lambda s: s.first_valid_index())})
print("all-NaN tickers (would wipe out their sphere):", all_nan)
print(report.sort_values('coverage').to_string(float_format=lambda x: f"{x:.1%}"))

assert not all_nan, f"remove these tickers: {all_nan}"
assert (stale < 0.10).all(), f"suspiciously stale tickers: {stale[stale >= 0.10].index.tolist()}"

all-NaN tickers (would wipe out their sphere): []
        coverage  stale_frac first_valid
Ticker                                  
ADI       100.0%        0.1%  2021-11-01
TSM       100.0%        0.2%  2021-11-01
TOELY     100.0%        0.1%  2021-11-01
TER       100.0%        0.2%  2021-11-01
STM       100.0%        0.6%  2021-11-01
SNPS      100.0%        0.1%  2021-11-01
QCOM      100.0%        0.0%  2021-11-01
ON        100.0%        0.4%  2021-11-01
NXPI      100.0%        0.0%  2021-11-01
NVDA      100.0%        0.2%  2021-11-01
MU        100.0%        0.0%  2021-11-01
MRVL      100.0%        0.2%  2021-11-01
TXN       100.0%        0.3%  2021-11-01
MCHP      100.0%        0.4%  2021-11-01
KLAC      100.0%        0.0%  2021-11-01
INTC      100.0%        0.5%  2021-11-01
IFNNY     100.0%        0.4%  2021-11-01
GFS       100.0%        0.3%  2021-11-01
CDNS      100.0%        0.0%  2021-11-01
AVGO      100.0%        0.1%  2021-11-01
ASX       100.0%        1.9%  2021-11-01
ASML   

### Market-Mode Residualization

At hourly frequency, the top eigenmode of the correlation matrix is the common market/sector factor and dominates raw returns. Hard MP truncation to the significant components keeps essentially *only* this mode (k=1 at our T/N), which makes the denoised returns rank-1 — and a rank-1 series produces an **exactly symmetric** lead-lag matrix, destroying the directionality the TMDN needs.

We therefore invert the logic: instead of keeping the market mode, we **project it out** and model lead-lag structure on the idiosyncratic residuals. $$\tilde{Z} = Z - (Z v_1)v_1^T$$
The removed factor time series $F_t = Z_t v_1$ is kept aside — it becomes its own *systematic layer* of the multiplex network, cleanly separating "the sector moved" from "asset $i$ leads asset $j$."

In [7]:
window = 252  # 1 trading year of daily bars per estimation window
step = 5      # slide by 1 trading week
assets_num = total_returns.shape[1]  # derive from the actual universe instead of hardcoding

# Guard: eigendecomposition cannot handle NaNs — fail loudly with the offending tickers
bad_cols = total_returns.columns[total_returns.isna().any()].tolist()
assert not bad_cols, f"total_returns still contains NaNs in: {bad_cols}"

all_residual_windows = []   # idiosyncratic residuals -> lead-lag layer (Sparse VAR)
all_market_factors = []     # removed market mode -> systematic layer
window_ends = []            # timestamp labels for the temporal dimension of the TMDN

for i in range(window, len(total_returns), step):
    window_returns = total_returns.iloc[i-window:i]
    residuals, factors = fl.remove_market_mode(window_returns, n_modes=1)
    all_residual_windows.append(residuals)
    all_market_factors.append(factors)
    window_ends.append(window_returns.index[-1])

print(f"{len(all_residual_windows)} weekly windows of {window} days, {assets_num} assets")
print("market mode variance share of last window:",
      f"{1 - all_residual_windows[-1].var().mean():.1%}")

186 weekly windows of 252 days, 27 assets
market mode variance share of last window: 50.6%


### Breaking Symmetry in the Financial Layer

To transform a standard undirected correlation into a Directed Lead-Lag Network, we define the directed adjacency matrix $A^{(dir)}$ using a time-shifted correlation.

For any two assets $i$ and $j$, the directed edge weight $E_{i \to j}$ is calculated as:$$E_{i \to j}(t) = \text{corr}(R_{i, t}, R_{j, t+1})$$
Conversely, the influence of $j$ on $i$ is:$$E_{j \to i}(t) = \text{corr}(R_{j, t}, R_{i, t+1})$$
In this construction, $A^{(dir)}$ is asymmetric ($E_{i \to j} \neq E_{j \to i}$), representing the directional flow of information from a "leader" to a "lagger."

### Sparse VAR(4) on Idiosyncratic Residuals

We estimate the directed lead-lag network with a Sparse VAR of order $p=4$ (four daily lags), fit jointly:
$$X_t = \sum_{L=1}^{4} A_L^T X_{t-L} + \varepsilon_t$$
One Lasso regression per target asset yields four asymmetric adjacency matrices $A_1, \dots, A_4$ — one temporal layer per horizon, so an edge $i \to j$ in $A_L$ reads "asset $i$'s move predicts asset $j$'s move $L$ days later, controlling for the other lags."

At daily frequency with a 252-day window, each regression has $252 - 4 = 248$ samples against $27 \times 4 = 108$ predictors — a comfortable regime for the Lasso (unlike hourly, where session boundaries left ~2 usable targets per day and the network failed its placebo test).

In [8]:
# Sparse VAR(4) on the most recent residual window
n_lags = 4
alpha = 0.02  # tune with the placebo test below — pick the smallest alpha whose network beats shuffled data

last_residuals = all_residual_windows[-1]
adjacencies = fl.var_lasso(last_residuals, alpha=alpha, n_lags=n_lags)

for L, A in adjacencies.items():
    nz = (A.values != 0)
    print(f"lag {L}d: {nz.sum():3d} edges ({nz.mean():.1%} density), "
          f"max |coef| = {np.abs(A.values).max():.3f}")

# Strongest edges across all lags
stacked = pd.concat({L: A.stack() for L, A in adjacencies.items()}, names=['lag', 'leader', 'lagger'])
top = stacked.abs().sort_values(ascending=False).head(10)
print("\nstrongest lead-lag edges (leader -> lagger @ lag):")
for (L, i, j), _ in top.items():
    print(f"  {i:6} -> {j:6} @ {L}d  {stacked.loc[(L, i, j)]:+.4f}")

# Directed graph of the 1-day layer
G = nx.from_pandas_adjacency(adjacencies[1], create_using=nx.DiGraph)
G.remove_edges_from([(u, v) for u, v, w in G.edges(data='weight') if w == 0])
print(f"\nlag-1 network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

lag 1d: 445 edges (61.0% density), max |coef| = 0.277
lag 2d: 444 edges (60.9% density), max |coef| = 0.246
lag 3d: 473 edges (64.9% density), max |coef| = 0.253
lag 4d: 419 edges (57.5% density), max |coef| = 0.471

strongest lead-lag edges (leader -> lagger @ lag):
  STM    -> TER    @ 4d  -0.4710
  ADI    -> ADI    @ 1d  +0.2765
  TXN    -> NXPI   @ 4d  +0.2757
  TER    -> ON     @ 3d  -0.2533
  TXN    -> AMKR   @ 4d  -0.2491
  ASX    -> SNPS   @ 2d  -0.2457
  AMAT   -> ADI    @ 3d  -0.2436
  ON     -> INTC   @ 1d  -0.2350
  ON     -> AMKR   @ 4d  +0.2341
  TXN    -> STM    @ 1d  +0.2260

lag-1 network: 27 nodes, 445 edges


### Placebo Test: Is the Network Real?

A Lasso will happily produce a sparse "network" from pure noise, so edge counts alone prove nothing. The control: **shuffle the time order of the residuals** (destroying any temporal lead-lag structure while preserving the contemporaneous cross-sectional correlations) and refit. If the real network isn't clearly denser / stronger than the shuffled ones, the edges are noise artifacts — this is exactly how the hourly pipeline was caught fitting microstructure. Use the smallest $\alpha$ whose real-to-placebo edge ratio is meaningfully above 1, and confirm with out-of-sample predictive $R^2 > 0$ on a held-out tail.

In [9]:
# Placebo: refit VAR(4) on time-shuffled residuals and compare edge counts,
# then check out-of-sample predictive R^2 on the most recent 20% of the window.
rng = np.random.default_rng(0)
n_placebos = 5

def edge_count(adj):
    return sum((A.values != 0).sum() for A in adj.values())

print(f"{'alpha':>7} {'real':>6} {'placebo':>8} {'ratio':>6} {'OOS R^2':>9}")
for a in (0.01, 0.02, 0.03, 0.05):
    real_edges = edge_count(fl.var_lasso(last_residuals, alpha=a, n_lags=n_lags))

    placebo_edges = []
    for _ in range(n_placebos):
        shuffled = last_residuals.sample(frac=1, random_state=rng.integers(1e9)).reset_index(drop=True)
        placebo_edges.append(edge_count(fl.var_lasso(shuffled, alpha=a, n_lags=n_lags)))

    # out-of-sample: fit on first 80% of the window, predict the rest
    split = int(len(last_residuals) * 0.8)
    train, test = last_residuals.iloc[:split], last_residuals.iloc[split - n_lags:]
    adj_tr = fl.var_lasso(train, alpha=a, n_lags=n_lags)
    Y = test.iloc[n_lags:]
    X = np.hstack([test.shift(L).iloc[n_lags:].values for L in range(1, n_lags + 1)])
    coefs = np.vstack([adj_tr[L].values for L in range(1, n_lags + 1)])
    pred = X @ coefs
    r2 = 1 - ((Y.values - pred) ** 2).sum() / (Y.values ** 2).sum()

    ratio = real_edges / max(np.mean(placebo_edges), 1)
    print(f"{a:7} {real_edges:6d} {np.mean(placebo_edges):8.0f} {ratio:6.2f} {r2:+9.4f}")

print("\nratio >> 1 and OOS R^2 > 0  ->  network reflects real temporal structure")
print("ratio ~= 1 or  OOS R^2 < 0  ->  edges are noise; raise alpha or rethink the layer")

  alpha   real  placebo  ratio   OOS R^2
   0.01   2255     2261   1.00   -0.8639
   0.02   1781     1800   0.99   -0.4916
   0.03   1416     1434   0.99   -0.3166
   0.05    887      895   0.99   -0.1578

ratio >> 1 and OOS R^2 > 0  ->  network reflects real temporal structure
ratio ~= 1 or  OOS R^2 < 0  ->  edges are noise; raise alpha or rethink the layer


## Volatility Spillover Layer (Diebold–Yilmaz)

Return lead-lag failed the placebo test at both hourly and daily frequency: cross-predictability of *returns* among liquid semis has been arbitraged away. **Volatility** is different — you cannot directly arbitrage "asset $j$ will be turbulent tomorrow", so vol predictability survives (log-vol lag-1 autocorrelation ≈ 0.4 vs ≈ 0 for returns), and *risk transmission is what contagion actually is*.

**Volatility proxy — Parkinson (range-based) estimator:** $$\sigma_t^2 = \frac{\ln(H_t/L_t)^2}{4\ln 2}$$ The intraday high–low range is ~5× more efficient than $|r_t|$ as a daily vol estimator. We work with $\ln \sigma_t$, which is approximately Gaussian (Andersen et al.).

**Connectedness — generalized FEVD (Diebold–Yilmaz 2012):** fit the Sparse VAR(4) on log-vols, then compute the forecast-error variance decomposition: $\theta_{ij}(H)$ = the share of asset $i$'s $H$-day-ahead forecast-error variance attributable to shocks originating in asset $j$. The row-normalized $\theta$ is a directed, weighted network:
- **TO$_j$** $= \sum_{i \ne j} \theta_{ij}$ — what $j$ exports (systemic *transmitters*)
- **FROM$_i$** $= \sum_{j \ne i} \theta_{ij}$ — what $i$ imports (the *vulnerable*)
- **NET** $=$ TO $-$ FROM, and the **total connectedness index** (mean off-diagonal share) — a single number per window that spikes in crises: the natural "temperature" of this TMDN layer.

Validation (walk-forward, train-window standardization): own-lags AR(4) achieves OOS $R^2$ up to +0.35 per window (+0.31 full sample) and the cross-asset VAR beats it in several regimes at $\alpha = 0.05$ — unlike returns, where every configuration was negative.

In [10]:
# Parkinson log-volatility from daily High/Low for the full universe
all_tickers = [t for d in spheres.values() for t in d]
ohlc = yf.download(all_tickers, start=START, interval=INTERVAL, prepost=False, progress=False)
log_vol = fl.parkinson_log_vol(ohlc['High'], ohlc['Low']).ffill().dropna()

print("log_vol:", log_vol.shape)
print("mean lag-1 autocorrelation:", log_vol.apply(lambda s: s.autocorr(1)).mean().round(3))

df = log_vol.reset_index()
fig = px.line(df, x=df.columns[0], y=log_vol.columns,
              title='Parkinson log-volatility (daily)',
              labels={'value': 'ln σ', 'variable': 'Ticker'})
fig.show()

log_vol: (1180, 27)
mean lag-1 autocorrelation: 0.396


In [11]:
# Validation: walk-forward OOS R^2 — does the cross-asset network beat own-lags AR?
# Standardize the test segment with TRAIN stats (test-stats standardization leaks regime info).
from sklearn.linear_model import Lasso

vol_alpha, n_lags = 0.05, 4

def _design(df, p):
    Y = df.iloc[p:]
    X = np.hstack([df.shift(L).iloc[p:].values for L in range(1, p + 1)])
    return X, Y.values

def walk_forward_r2(win, p=n_lags, alpha=vol_alpha, split_frac=0.8):
    split = int(len(win) * split_frac)
    mu, sd = win.iloc[:split].mean(), win.iloc[:split].std()
    z = (win - mu) / sd
    train, test = z.iloc[:split], z.iloc[split - p:]

    # full cross-asset VAR
    adj = fl.var_lasso(train, alpha=alpha, n_lags=p)
    coefs = np.vstack([adj[L].values for L in range(1, p + 1)])
    Xte, Yte = _design(test, p)
    r2_full = 1 - ((Yte - Xte @ coefs) ** 2).sum() / (Yte ** 2).sum()

    # own-lags AR benchmark (per-asset OLS)
    r2s = []
    for c in win.columns:
        Xtr, Ytr = _design(train[[c]], p)
        beta, *_ = np.linalg.lstsq(Xtr, Ytr[:, 0], rcond=None)
        Xc, Yc = _design(test[[c]], p)
        r2s.append(1 - ((Yc[:, 0] - Xc @ beta) ** 2).sum() / (Yc[:, 0] ** 2).sum())
    return r2_full, np.mean(r2s)

print(f"{'window end':>12} {'VAR (cross)':>12} {'AR (own)':>10} {'delta':>8}")
for end in range(window, len(log_vol) + 1, 126):
    win = log_vol.iloc[end - window:end]
    r2f, r2a = walk_forward_r2(win)
    print(f"{str(win.index[-1].date()):>12} {r2f:+12.4f} {r2a:+10.4f} {r2f - r2a:+8.4f}")
r2f, r2a = walk_forward_r2(log_vol)
print(f"{'FULL':>12} {r2f:+12.4f} {r2a:+10.4f} {r2f - r2a:+8.4f}")

  window end  VAR (cross)   AR (own)    delta
  2022-10-31      -0.0516    +0.0537  -0.1053
  2023-05-03      +0.2687    +0.2059  +0.0628
  2023-11-01      -0.0750    -0.0017  -0.0733
  2024-05-03      -0.0038    +0.0595  -0.0633
  2024-11-01      +0.0109    +0.0283  -0.0174
  2025-05-07      +0.3466    +0.3471  -0.0005
  2025-11-05      +0.0157    +0.0447  -0.0291
  2026-05-08      +0.2557    +0.2276  +0.0281
        FULL      +0.3121    +0.3068  +0.0052


In [12]:
# Rolling Diebold-Yilmaz connectedness: one directed spillover network per weekly window.
# This IS the financial layer of the TMDN: supra-adjacency A[i, j, t] = theta_t[i, j].
fevd_horizon = 10  # days

vol_tables, vol_summaries, vol_total, vol_ends = [], [], [], []
for i in range(window, len(log_vol), step):
    win = log_vol.iloc[i-window:i]
    z = (win - win.mean()) / win.std()
    adj = fl.var_lasso(z, alpha=vol_alpha, n_lags=n_lags)
    coefs = np.vstack([adj[L].values for L in range(1, n_lags + 1)])
    X, Yv = _design(z, n_lags)
    resid = pd.DataFrame(Yv - X @ coefs, columns=z.columns)
    table, summary = fl.fevd_connectedness(adj, resid, horizon=fevd_horizon)
    vol_tables.append(table)
    vol_summaries.append(summary)
    vol_total.append(summary.attrs['total'])
    vol_ends.append(win.index[-1])

connectedness = pd.Series(vol_total, index=pd.DatetimeIndex(vol_ends), name='total_connectedness')
fig = px.line(connectedness, title='Total Volatility Connectedness Index (rolling 252d, weekly steps)',
              labels={'value': 'total connectedness', 'index': 'window end'})
fig.show()

latest = vol_summaries[-1]
print(f"latest window ({vol_ends[-1].date()}): total connectedness = {vol_total[-1]:.1%}\n")
print("top systemic TRANSMITTERS (NET > 0):")
print(latest.sort_values('NET', ascending=False).head(5).to_string(float_format=lambda x: f"{x:.3f}"))
print("\nmost VULNERABLE (NET < 0):")
print(latest.sort_values('NET').head(5).to_string(float_format=lambda x: f"{x:.3f}"))

# strongest directed spillover edges of the latest network
off = vol_tables[-1].where(~np.eye(len(latest), dtype=bool), 0.0)
print("\nstrongest directed spillovers (theta share of receiver's FEV), source -> receiver:")
for (i, j), v in off.stack().sort_values(ascending=False).head(8).items():
    print(f"  {j:6} -> {i:6}  {v:.3f}")

latest window (2026-07-13): total connectedness = 81.2%

top systemic TRANSMITTERS (NET > 0):
          TO  FROM   NET
Ticker                  
STM    1.212 0.864 0.348
AMAT   1.216 0.877 0.339
ASX    1.195 0.872 0.323
TER    1.139 0.871 0.268
LRCX   1.117 0.866 0.251

most VULNERABLE (NET < 0):
          TO  FROM    NET
Ticker                   
INTC   0.314 0.719 -0.405
TOELY  0.412 0.724 -0.312
SNPS   0.409 0.664 -0.255
QCOM   0.539 0.784 -0.246
CDNS   0.508 0.724 -0.217

strongest directed spillovers (theta share of receiver's FEV), source -> receiver:
  CDNS   -> SNPS    0.135
  SNPS   -> CDNS    0.109
  STM    -> IFNNY   0.093
  AMAT   -> LRCX    0.085
  AMAT   -> KLAC    0.078
  LRCX   -> KLAC    0.078
  LRCX   -> AMAT    0.077
  MCHP   -> ON      0.075
